In [ ]:
# Install dependencies for local Python 3.10 and Colab


### Set openai key

In [ ]:
import openai
import os
# Set OPENAI_API_KEY in environment variables, do not hardcode it here


In [ ]:
import os
import openai
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError(
        "Set OPENAI_API_KEY in your environment before running this notebook. "
        "Do not paste secrets into notebook cells."
    )

openai.api_key = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")


### Download pdf

In [ ]:
from pathlib import Path

pdf_path = Path("article_increasingreturns.pdf")
if pdf_path.exists():
    print(f"Using existing PDF: {pdf_path.resolve()}")
else:
    sample_pdf_path = Path("sample_article.pdf")
    sample_pdf_path.write_bytes(b"""%PDF-1.4
1 0 obj
<< /Type /Catalog /Pages 2 0 R >>
endobj
2 0 obj
<< /Type /Pages /Kids [3 0 R] /Count 1 >>
endobj
3 0 obj
<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents 4 0 R /Resources << /Font << /F1 5 0 R >> >> >>
endobj
4 0 obj
<< /Length 62 >>
stream
BT /F1 18 Tf 72 740 Td (Sample PDF for notebook execution.) Tj ET
endstream
endobj
5 0 obj
<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>
endobj
xref
0 6
0000000000 65535 f 
0000000010 00000 n 
0000000061 00000 n 
0000000117 00000 n 
0000000221 00000 n 
0000000332 00000 n 
trailer
<< /Root 1 0 R /Size 6 >>
startxref
404
%%EOF
""")
    pdf_path = sample_pdf_path
    print(f"Created fallback sample PDF: {pdf_path.resolve()}")


In [ ]:
from pathlib import Path

for path in sorted(Path('.').glob('*')):
    print(path)


### Extract text from pdf document

In [ ]:
from langchain_classic.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(pdf_path))
pages = loader.load_and_split()
print(f"Loaded {len(pages)} page chunks from {pdf_path}")


In [ ]:
pages[0:2]


### Summarize the text using Langchain

In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI


In [ ]:
llm = ChatOpenAI(temperature=0.1, model_name=OPENAI_MODEL, api_key=openai.api_key)
chain = load_summarize_chain(llm, chain_type="stuff")


In [ ]:
res = chain.invoke(pages[0:2])


In [ ]:
print(res["output_text"])


### 1. stuff document chain method

In [ ]:
from langchain_classic.chains.combine_documents.stuff import StuffDocumentsChain
from langchain_classic.chains.llm import LLMChain
from langchain_classic.prompts import PromptTemplate


In [ ]:
# Define prompt
prompt_template = """Write a concise summary in a maximum of 3 bullets of the following text enclosed within three backticks:
```{text}```
CONCISE SUMMARY:"""
prompt = PromptTemplate.from_template(prompt_template)

# Define LLM chain
llm = ChatOpenAI(temperature=0, model_name=OPENAI_MODEL, api_key=openai.api_key)
llm_chain = LLMChain(llm=llm, prompt=prompt)

# Define StuffDocumentsChain
stuff_chain = StuffDocumentsChain(llm_chain=llm_chain, document_variable_name="text")

res = stuff_chain.invoke(pages[0:3])


In [ ]:
print(res["output_text"])


### 2. Map-Reduce chain method


#### Reference: https://medium.com/@abonia/summarization-with-langchain-b3d83c030889

In [ ]:
from langchain_classic.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain_text_splitters import CharacterTextSplitter


In [ ]:
map_prompt_template = """
                      Write a summary of this chunk of text that includes the main points and any important details.
                      {text}
                      """

map_prompt = PromptTemplate(template=map_prompt_template, input_variables=["text"])

combine_prompt_template = """
                      Write a concise summary of the following text delimited by triple backquotes.
                      Return your response in bullet points which covers the key points of the text.
                      ```{text}```
                      BULLET POINT SUMMARY:
                      """

combine_prompt = PromptTemplate(
    template=combine_prompt_template, input_variables=["text"]
)


In [ ]:
map_reduce_chain = load_summarize_chain(
    llm,
    chain_type="map_reduce",
    map_prompt=map_prompt,
    combine_prompt=combine_prompt,
    return_intermediate_steps=True,
)


In [ ]:
map_reduce_outputs = map_reduce_chain({"input_documents": pages[0:3]})


In [ ]:
map_reduce_outputs


# Extra Credit

### 3. Refine method

In [ ]:
question_prompt_template = """
                  Please provide a summary of the following text.
                  TEXT: {text}
                  SUMMARY:
                  """

question_prompt = PromptTemplate(
    template=question_prompt_template, input_variables=["text"]
)

refine_prompt_template = """
              Write a concise summary of the following text delimited by triple backquotes.
              Return your response in bullet points which covers the key points of the text.
              ```{text}```
              BULLET POINT SUMMARY:
              """

refine_prompt = PromptTemplate(
    template=refine_prompt_template, input_variables=["text"]
)


In [ ]:
refine_chain = load_summarize_chain(
    llm,
    chain_type="refine",
    question_prompt=question_prompt,
    refine_prompt=refine_prompt,
    return_intermediate_steps=True,
)


In [ ]:
refine_outputs = refine_chain({"input_documents": pages[0:3]})


In [ ]:
refine_outputs
